In [1]:
# ===== P4 CELL 1: Connect Google Drive =====
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ===== P4 CELL 2: Load settings and the label/identifier files =====
import os, json, hashlib
import numpy as np
import pandas as pd

BASE = os.environ.get("BETH_BASE", "/content/drive/MyDrive/BETH_Triage")
FEAT_DIR = os.path.join(BASE, "02_data_processed", "features")
SPLIT_DIR = os.path.join(BASE, "02_data_processed", "splits")
RES_DIR = os.path.join(BASE, "05_results")
os.makedirs(SPLIT_DIR, exist_ok=True)

with open(os.path.join(BASE, "config.json")) as fh:
    CONFIG = json.load(fh)
proto = CONFIG["protocol"]
with open(os.path.join(BASE, proto["file"]), "rb") as fh:
    assert hashlib.sha256(fh.read()).hexdigest() == proto["sha256"], "Protocol file changed - STOP"
PLAN = proto["plan"]
print("Protocol", proto["version"], "verified unchanged")

meta = {s: pd.read_pickle(os.path.join(FEAT_DIR, f"meta_{s}.pkl")) for s in ["train", "val", "test"]}
for s, m in meta.items():
    print(f"{s:5s}: {len(m):,} rows, {m['process_key'].nunique()} processes, hosts: {sorted(m['hostName'].unique())}")

Protocol v1 verified unchanged
train: 763,144 rows, 1179 processes, hosts: ['ip-10-100-1-120', 'ip-10-100-1-173', 'ip-10-100-1-28', 'ip-10-100-1-34', 'ip-10-100-1-55', 'ip-10-100-1-57', 'ip-10-100-1-79', 'ubuntu']
val  : 188,967 rows, 244 processes, hosts: ['ip-10-100-1-129', 'ip-10-100-1-165', 'ip-10-100-1-169', 'ubuntu']
test : 188,967 rows, 198 processes, hosts: ['ip-10-100-1-217']


In [3]:
# ===== P4 CELL 3: Leave-one-validation-host-out calibration folds (for hypothesis H1) =====
val = meta["val"]
hosts = sorted(val["hostName"].unique())
folds, rows = [], []
for held_out in hosts:
    calib_hosts = [h for h in hosts if h != held_out]
    calib_mask = val["hostName"].isin(calib_hosts).values
    n_cal_rows = int(calib_mask.sum())
    n_cal_procs = int(val.loc[calib_mask, "process_key"].nunique())
    folds.append({"held_out_host": held_out, "calibration_hosts": calib_hosts})
    rows.append({
        "held_out_host": held_out,
        "calibration_rows": n_cal_rows,
        "held_out_rows": int((~calib_mask).sum()),
        "calibration_processes": n_cal_procs,
        "smallest_event_alpha": round(1 / (n_cal_rows + 1), 6),
        "smallest_process_alpha": round(1 / (n_cal_procs + 1), 4),
    })
final_procs = int(val["process_key"].nunique())
rows.append({"held_out_host": "(none: final calibration = all 4 hosts)",
             "calibration_rows": len(val), "held_out_rows": 0,
             "calibration_processes": final_procs,
             "smallest_event_alpha": round(1 / (len(val) + 1), 6),
             "smallest_process_alpha": round(1 / (final_procs + 1), 4)})
fold_df = pd.DataFrame(rows)
with open(os.path.join(SPLIT_DIR, "calibration_folds.json"), "w") as fh:
    json.dump({"folds": folds, "final_calibration_hosts": hosts}, fh, indent=2)
fold_df.to_csv(os.path.join(RES_DIR, "P4_calibration_folds.csv"), index=False)
pd.set_option("display.width", 200)
print(fold_df.to_string(index=False))

print("\nWhich alphas can be reached at process level (final calibration)?")
for a in PLAN["alphas"]:
    ok = a >= 1 / (final_procs + 1)
    print(f"   alpha = {a:<6} -> {'reachable' if ok else 'NOT reachable (reported as such)'}")

                          held_out_host  calibration_rows  held_out_rows  calibration_processes  smallest_event_alpha  smallest_process_alpha
                        ip-10-100-1-129            173941          15026                    202              0.000006                  0.0049
                        ip-10-100-1-165            172957          16010                    203              0.000006                  0.0049
                        ip-10-100-1-169            174332          14635                    207              0.000006                  0.0048
                                 ubuntu             45671         143296                    120              0.000022                  0.0083
(none: final calibration = all 4 hosts)            188967              0                    244              0.000005                  0.0041

Which alphas can be reached at process level (final calibration)?
   alpha = 0.001  -> NOT reachable (reported as such)
   alpha = 0.005  -> reacha

In [4]:
# ===== P4 CELL 4: Secondary check - real-record test mixes (row indices only, no data changed) =====
test = meta["test"]
normal_idx = np.where(test["evil"].values == 0)[0]
attack_idx = np.where(test["evil"].values == 1)[0]
attack_by_time = attack_idx[np.argsort(test["timestamp"].values[attack_idx], kind="stable")]
mix = PLAN["secondary_mix"]

mixes, summary = {}, []
for pi in mix["prevalences"]:
    k = int(round(pi / (1 - pi) * len(normal_idx)))
    for seed in mix["seeds"]:
        rng = np.random.default_rng(1000 + seed)
        rand_att = np.sort(rng.choice(attack_idx, size=k, replace=False))
        start = int(rng.integers(0, len(attack_by_time) - k + 1))
        win_att = np.sort(attack_by_time[start:start + k])
        for kind, att in [("random_rows", rand_att), ("contiguous_time_window", win_att)]:
            idx = np.concatenate([normal_idx, att])
            mixes[f"{kind}|pi={pi}|seed={seed}"] = idx
            summary.append({"sampling": kind, "attack_share_target": pi, "seed": seed,
                            "rows": len(idx), "attack_rows": len(att),
                            "attack_share_actual": round(len(att) / len(idx), 4),
                            "attack_processes": len(set(test["process_key"].values[att]))})
np.savez_compressed(os.path.join(SPLIT_DIR, "secondary_mixes.npz"), **mixes)
mix_df = pd.DataFrame(summary)
mix_df.to_csv(os.path.join(RES_DIR, "P4_secondary_mixes.csv"), index=False)
print(mix_df.groupby(["sampling", "attack_share_target"])
      .agg(mixes=("seed", "size"), rows=("rows", "first"), attack_rows=("attack_rows", "first"),
           share=("attack_share_actual", "first"), attack_processes_min=("attack_processes", "min"),
           attack_processes_max=("attack_processes", "max")).to_string())
print(f"\nSaved {len(mixes)} mixes (row indices into the unmodified test file).")

                                            mixes   rows  attack_rows  share  attack_processes_min  attack_processes_max
sampling               attack_share_target                                                                              
contiguous_time_window 0.01                    10  30843          308   0.01                     1                     9
                       0.05                    10  32142         1607   0.05                     1                     1
                       0.10                    10  33928         3393   0.10                     1                    29
random_rows            0.01                    10  30843          308   0.01                    12                    24
                       0.05                    10  32142         1607   0.05                    39                    56
                       0.10                    10  33928         3393   0.10                    67                    83

Saved 60 mixes (row indices int

In [5]:
# ===== P4 CELL 5: Integrity checks =====
loaded = np.load(os.path.join(SPLIT_DIR, "secondary_mixes.npz"))
for key in list(loaded.files)[:60]:
    idx = loaded[key]
    assert len(np.unique(idx)) == len(idx), f"duplicate rows in {key}"
    assert idx.min() >= 0 and idx.max() < len(test), f"index out of range in {key}"
    assert set(normal_idx).issubset(set(idx)), f"normal rows missing in {key}"
with open(os.path.join(SPLIT_DIR, "calibration_folds.json")) as fh:
    f = json.load(fh)
for fold in f["folds"]:
    assert fold["held_out_host"] not in fold["calibration_hosts"], "held-out host leaked into calibration"
assert set(meta["val"]["hostName"]).isdisjoint(set(meta["test"]["hostName"])), "validation and test share a host"
print("All checks passed:")
print("  - every mix keeps all", f"{len(normal_idx):,}", "normal test rows, no duplicates, valid indices")
print("  - no held-out host appears in its own calibration set")
print("  - validation and test hosts are completely separate")

All checks passed:
  - every mix keeps all 30,535 normal test rows, no duplicates, valid indices
  - no held-out host appears in its own calibration set
  - validation and test hosts are completely separate
